# Spatial Projection

This notebook covers projecting point and linear geometries onto a linearly referenced network.

In [ ]:
import sys
sys.path.insert(0, '.')
from _setup import lr, roadways, crashes, pavement

## Projecting Point Geometries

When you have point data with only spatial coordinates (no route or milepost information), you can project those points onto your linearly referenced network. This is useful for integrating external data sources like GPS points, sensor locations, or crash data that hasn't been linearly referenced yet.

First, we'll create some sample point geometries to simulate data without LRS information:

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# First, ensure roadways have M-enabled geometries for projection
roadways_spatial = roadways.lr.add_geom_m()

# Create sample point features (e.g., sign locations) with only X/Y coordinates
# These points are near the roadway network but don't have route or milepost info
points = gpd.GeoDataFrame({
    'sign_id': [1, 2, 3],
    'sign_type': ['Speed Limit', 'Stop', 'Yield'],
    'geometry': [
        Point(-122.3321, 47.6062),  # Near a roadway
        Point(-122.3315, 47.6065),  # Near a roadway
        Point(-122.3350, 47.6070)   # Near a roadway
    ]
}, crs=roadways.crs)

# Project points onto the roadway network
# buffer=50 searches for roads within 50 meters of each point
projected = roadways_spatial.lr.project(points, buffer=50, nearest=True, dropna=True)

print("Points after projection (now have route and loc columns):")
print(projected[['sign_id', 'sign_type', 'route', 'loc', 'project_distance']])

## Projecting Linear Geometries

When you need to match linear geometries from one dataset to another (e.g., conflating road data from different sources), you can use `parallel_project_hausdorff()`. This function uses Hausdorff distance to find the best geometric match between linear features.

This is useful for:
- Conflating road inventories from different agencies
- Matching GPS traces to a road network
- Integrating pavement surveys onto a master road system

Here's an example of creating alternative road segments and projecting them onto our master roadway network:

In [ ]:
from linref.ext.spatial import parallel_project_hausdorff
from shapely.geometry import LineString

# Create sample linear segments from another source (e.g., county road inventory)
# These have geometries but no route IDs or mileposts from our system
other_segments = gpd.GeoDataFrame({
    'segment_id': ['A', 'B'],
    'data_source': ['County', 'County'],
    'geometry': [
        LineString([(-122.3320, 47.6061), (-122.3318, 47.6063)]),  # Should match a roadway
        LineString([(-122.3348, 47.6069), (-122.3346, 47.6071)])   # Should match another roadway
    ]
}, crs=roadways.crs)

# Project these segments onto our master roadway network
# The function finds the best geometric match using Hausdorff distance
projected_segments = parallel_project_hausdorff(
    target=roadways_spatial,      # Our master roadway network with M-geometries
    projected=other_segments,      # Segments to be matched
    buffer=100,                    # Search within 100m of segment endpoints
    max_distance=50,               # Only accept matches with similarity < 50m
    match=1,                       # Return best match only
    densify=0.1                    # Add vertices for accurate matching
)

print("Segments after projection (now have route, beg, and end from our LRS):")
print(projected_segments[['segment_id', 'data_source', 'route', 'beg', 'end']])